# Teste — módulo de coleta

**Kernel:** conda `moradinha`  
**Diretório de trabalho:** deve ser `e:/Doutorado/UFABC/moradinha/` (raiz do projeto)

In [ ]:
import logging
import sys
from pathlib import Path

# Garante que o diretório raiz está no path
ROOT = Path("e:/Doutorado/UFABC/moradinha")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Logging legível no notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

print("OK")

In [ ]:
from modulo_coleta.orquestrador import coletar_municipio

## Parâmetros

Edite o código IBGE e os grupos desejados antes de rodar.

In [ ]:
CODIGO_IBGE   = "2701407"          # Campo Alegre-AL (usado nos testes anteriores)
NOME_PASTA    = "al_campo_alegre"  # None = derivado automaticamente via geobr
GRUPOS        = [1, 2, 3, 4, 5]   # remova grupos que não quer coletar
BASE_DIR      = ROOT / "data"
FORCAR        = False              # True = reprocessa mesmo que arquivos já existam

In [ ]:
resultados = coletar_municipio(
    codigo_ibge=CODIGO_IBGE,
    grupos=GRUPOS,
    base_dir=BASE_DIR,
    nome_municipio=NOME_PASTA,
    forcar=FORCAR,
)

## Resumo dos resultados

In [ ]:
from modulo_coleta.orquestrador import GRUPOS_DISPONIVEIS

for grupo, res in resultados.items():
    if grupo == "mapa":
        continue
    nome  = GRUPOS_DISPONIVEIS.get(grupo, f"grupo{grupo}")
    status = res.get("status", "?")
    camadas = res.get("camadas", [])
    msg    = res.get("mensagem", "")
    icone  = "✅" if status == "ok" else ("⏭️" if status == "pulado" else "❌")
    print(f"{icone} Grupo {grupo} ({nome}): {status}")
    if camadas:
        print(f"   camadas: {camadas}")
    if status != "ok":
        print(f"   msg: {msg}")

## Mapa de coleta

In [ ]:
from IPython.display import Image, display

mapa_path = resultados.get("mapa")
if mapa_path and Path(mapa_path).exists():
    display(Image(filename=mapa_path, width=800))
else:
    print("Mapa não gerado.")

## Inspecionar o DuckDB

Célula opcional para checar as tabelas geradas.

In [ ]:
from modulo_coleta.utils.db_utils import abrir_conexao, listar_tabelas

db_path = BASE_DIR / "processed" / NOME_PASTA / f"{NOME_PASTA}.duckdb"
conn = abrir_conexao(db_path)
tabelas = listar_tabelas(conn)
print(f"Tabelas no banco ({NOME_PASTA}):")
for t in tabelas:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} registros")
conn.close()